# Parameter uncertainty — Monte-Carlo IIV propagation

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

The dataset stores inter-individual variability (`iiv_cv_percent`) on its high-uncertainty kill/resistance terms so a ~90%-CV term cannot masquerade as a point estimate. `simulate_ensemble` makes that stored uncertainty *flow into the prediction*: parameters with an IIV CV are sampled lognormally and the tumor-size / TGI-metric / OS distributions are summarized as bands.

This complements the virtual-trial divergence view: that captures **model-selection** uncertainty; this captures **parameter** uncertainty.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
t = np.linspace(0, 104, 209)
ens = onkos.simulate_ensemble(ds, 'resistance.claret_2009.tgi',
                             context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.0, t=t, n=400, seed=0)
for k, v in ens.metrics.items():
    print(f"{k:<24} median {v['median']:.3f}  [{v['lo']:.3f}, {v['hi']:.3f}]")

In [ ]:
b, o = ens.tumor_size, ens.os_curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.fill_between(t, b.lo, b.hi, alpha=0.25); ax1.plot(t, b.median)
ax1.set(title='tumor size (5-95% band)', xlabel='weeks', ylabel='SLD (mm)')
ax2.fill_between(t, o.lo, o.hi, alpha=0.25, color='green'); ax2.plot(t, o.median, color='green')
ax2.axhline(0.5, ls=':', color='grey'); ax2.set(title='population OS (5-95% band)', xlabel='weeks', ylabel='S(t)')
# bands are ordered and the no-IIV case is degenerate
assert np.all(b.lo <= b.median + 1e-9) and np.all(b.median <= b.hi + 1e-9)

In [ ]:
# A model with no reported IIV (a growth law) yields a degenerate, zero-width band.
g = onkos.simulate_ensemble(ds, 'growth_laws.gompertz', context={'tumor_type': 'NSCLC'}, drug_effect=0.0, n=50)
print('growth-law band width (should be ~0):', float(np.max(g.tumor_size.hi - g.tumor_size.lo)))
assert np.allclose(g.tumor_size.lo, g.tumor_size.hi)

# Sampling is reproducible given a seed.
a = onkos.simulate_ensemble(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'}, n=100, seed=42)
b2 = onkos.simulate_ensemble(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'}, n=100, seed=42)
assert np.allclose(a.tumor_size.median, b2.tumor_size.median)
print('reproducible:', True)